In [1]:
import pandas as pd
import os
import numpy as np
from datetime import datetime, timedelta
import re

In [2]:
pd.set_option('display.max_columns', None)

#### <span style="color:#FF6347;">**READ**</span> 

In [3]:
# file_path = "P:/Dataset/MO-DBT-data-curation/Cancer/mo_cancer.xlsx"
file_path = os.path.join("/Users/tracyliu/Library/CloudStorage/OneDrive-UniversityofPittsburgh/R01-MO-DBT/MO-DBT-data-curation/Data", "Cancer", "mo_cancer" + ".xlsx")
mo_cancer = pd.read_excel(file_path)
file_path = os.path.join("/Users/tracyliu/Library/CloudStorage/OneDrive-UniversityofPittsburgh/R01-MO-DBT/MO-DBT-data-curation/Data", "Control", "normal_candidate" + ".xlsx")
normal_candidate = pd.read_excel(file_path)

file_path = os.path.join("/Users/tracyliu/Library/CloudStorage/OneDrive-UniversityofPittsburgh/R01-MO-DBT/MO-DBT-data-curation/Data", "Cancer", "mo_cancer_ehr_@_index_1" + ".xlsx")
mo_cancer_ehr = pd.read_excel(file_path)
file_path = os.path.join("/Users/tracyliu/Library/CloudStorage/OneDrive-UniversityofPittsburgh/R01-MO-DBT/MO-DBT-data-curation/Data", "Control", "normal_candidate_ehr_@_index_1" + ".xlsx")
normal_candidate_ehr = pd.read_excel(file_path)

# **Step 1. BI-RADS**

In [4]:
# ── STEP 1: Bilateral BIRADS combo at INDEX-1 ─────────────────────────────────
SHORT = {'1 - Negative': '1', '2 - Benign finding': '2'}

c1 = mo_cancer[mo_cancer['Index'] == 'INDEX-1'].copy()
n1 = normal_candidate[normal_candidate['Index'] == 'INDEX-1'].copy()

def get_bilateral_combo(df):
    df = df.copy()
    df['CAT'] = df['FINDING_CATEGORY'].map(SHORT)

    side_dedup = df.drop_duplicates(subset=['PATIENT_STUDY_ID', 'SIDE'], keep='first')

    valid = side_dedup[side_dedup['SIDE'].isin(['L', 'R'])]
    piv = valid.pivot_table(
        index='PATIENT_STUDY_ID', columns='SIDE', values='CAT', aggfunc='first'
    )
    piv.columns.name = None
    def combo(row):
        vals = sorted([v for v in row.dropna()])
        return '+'.join(vals) if vals else None
    piv['birads_combo'] = piv.apply(combo, axis=1)
    result = piv[['birads_combo']].reset_index()

    nan_side_pids = set(df['PATIENT_STUDY_ID']) - set(result['PATIENT_STUDY_ID'])
    if nan_side_pids:
        nan_side = (
            df[df['PATIENT_STUDY_ID'].isin(nan_side_pids)]
            .drop_duplicates(subset=['PATIENT_STUDY_ID'], keep='first')
            [['PATIENT_STUDY_ID', 'CAT']]
            .rename(columns={'CAT': 'birads_combo'})
        )
        result = pd.concat([result, nan_side], ignore_index=True)

    return result

c_combo = get_bilateral_combo(c1)
n_combo = get_bilateral_combo(n1)

# ── Distribution table ────────────────────────────────────────────────────────
label_map = {
    '2+2': '2+2 (both Benign)',
    '1+1': '1+1 (both Negative)',
    '1+2': '1+2 (mixed)',
    '2':   '2 only (unilateral) + N/A',
    '1':   '1 only (unilateral) + N/A',
}

def combo_table(df, label):
    n_total = df['PATIENT_STUDY_ID'].nunique()
    counts  = df['birads_combo'].value_counts(dropna=False)
    tbl = pd.DataFrame({
        'Combo': counts.index.map(lambda x: label_map.get(str(x), str(x))),
        'Count': counts.values,
        '%':     (counts.values / n_total * 100).round(1),
    })
    print(f"{'─'*45}")
    print(f"{label}  (n={n_total:,})")
    print(f"{'─'*45}")
    print(tbl.to_string(index=False))
    print()

combo_table(c_combo, 'MO Cancer')
combo_table(n_combo, 'Normal Candidate')

─────────────────────────────────────────────
MO Cancer  (n=253)
─────────────────────────────────────────────
                    Combo  Count    %
        2+2 (both Benign)    197 77.9
      1+1 (both Negative)     51 20.2
2 only (unilateral) + N/A      5  2.0

─────────────────────────────────────────────
Normal Candidate  (n=30,903)
─────────────────────────────────────────────
                    Combo  Count    %
        2+2 (both Benign)  17912 58.0
      1+1 (both Negative)   9461 30.6
1 only (unilateral) + N/A   3288 10.6
2 only (unilateral) + N/A    242  0.8



# **Step 2. Race**

In [5]:
c_race_cols = [c for c in mo_cancer_ehr.columns if c.startswith('RACE_')]
n_race_cols = [c for c in normal_candidate_ehr.columns if c.startswith('RACE_')]

In [6]:
ASIAN_GROUPS = {
    'KOREAN', 'OTHER ASIAN', 'VIETNAMESE', 'CHINESE', 'FILIPINO',
    'JAPANESE', 'INDIAN (ASIAN)'
}

PACIFIC_GROUPS = {
    'HAWAIIAN',  'OTHER PACIFIC ISLANDER'
}

def get_simplified_race(df):
    race_cols = [c for c in df.columns if c.startswith('RACE_')]
    df2 = df[race_cols].fillna(0)
    all_zero = df2.sum(axis=1) == 0
    race = df2.idxmax(axis=1).str.replace('RACE_', '')
    race[all_zero] = 'UNKNOWN'
    def simplify(r):
        if r == 'WHITE':  return 'WHITE'
        if r == 'BLACK':  return 'BLACK'
        if r == 'AMERICAN INDIAN/ALASKA NATIVE':  return 'AMERICAN INDIAN/ALASKA NATIVE'
        if r in PACIFIC_GROUPS: return 'HAWAIIAN/PACIFIC ISLANDER'
        if r in ASIAN_GROUPS: return 'ASIAN'
        return 'OTHER/UNKNOWN'
    return race.apply(simplify)

mo_cancer_ehr['race']        = get_simplified_race(mo_cancer_ehr)
normal_candidate_ehr['race'] = get_simplified_race(normal_candidate_ehr)

In [7]:
def race_summary(df, label):
    n_total = len(df)
    counts = df['race'].value_counts(dropna=False)
    tbl = pd.DataFrame({
        'Race':  counts.index,
        'Count': counts.values,
        '%':     (counts.values / n_total * 100).round(1),
    })
    print(f"{'─'*40}")
    print(f"{label}  (n={n_total:,})")
    print(f"{'─'*40}")
    print(tbl.to_string(index=False))
    print()

race_summary(mo_cancer_ehr,        'MO Cancer')
race_summary(normal_candidate_ehr, 'Normal Candidate')

────────────────────────────────────────
MO Cancer  (n=253)
────────────────────────────────────────
         Race  Count    %
        WHITE    234 92.5
        BLACK     11  4.3
        ASIAN      5  2.0
OTHER/UNKNOWN      3  1.2

────────────────────────────────────────
Normal Candidate  (n=30,901)
────────────────────────────────────────
                         Race  Count    %
                        WHITE  27833 90.1
                        BLACK   1496  4.8
                OTHER/UNKNOWN    784  2.5
                        ASIAN    753  2.4
AMERICAN INDIAN/ALASKA NATIVE     23  0.1
    HAWAIIAN/PACIFIC ISLANDER     12  0.0



In [8]:
mo_cancer_ehr['PATIENT_STUDY_ID'].nunique(), normal_candidate_ehr['PATIENT_STUDY_ID'].nunique()

(253, 30901)

# **Step 3. Age**

In [9]:
mo_cancer_ehr['PatientAge'].min(), mo_cancer_ehr['PatientAge'].max()

(38, 87)

In [10]:
AGE_BINS   = list(range(30, 90, 5))
AGE_LABELS = [f'{b}-{b+4}' for b in AGE_BINS[:-1]]

mo_cancer_ehr['age_bin']        = pd.cut(mo_cancer_ehr['PatientAge'],        bins=AGE_BINS, labels=AGE_LABELS, right=False)
normal_candidate_ehr['age_bin'] = pd.cut(normal_candidate_ehr['PatientAge'], bins=AGE_BINS, labels=AGE_LABELS, right=False)

In [11]:
def age_summary(df, label):
    n_total = len(df)
    counts = df['age_bin'].value_counts(dropna=False)
    tbl = pd.DataFrame({
        'AGE_LABELS':  counts.index,
        'Count': counts.values,
        '%':     (counts.values / n_total * 100).round(1),
    })
    print(f"{'─'*40}")
    print(f"{label}  (n={n_total:,})")
    print(f"{'─'*40}")
    print(tbl.to_string(index=False))
    print()

age_summary(mo_cancer_ehr,        'MO Cancer')
age_summary(normal_candidate_ehr, 'Normal Candidate')

────────────────────────────────────────
MO Cancer  (n=253)
────────────────────────────────────────
AGE_LABELS  Count    %
     55-59     45 17.8
     60-64     42 16.6
     65-69     41 16.2
     70-74     31 12.3
     50-54     26 10.3
     45-49     19  7.5
     75-79     16  6.3
     40-44     14  5.5
     80-84      9  3.6
       NaN      7  2.8
     35-39      3  1.2
     30-34      0  0.0

────────────────────────────────────────
Normal Candidate  (n=30,901)
────────────────────────────────────────
AGE_LABELS  Count    %
     60-64   4826 15.6
     65-69   4359 14.1
     55-59   4354 14.1
     50-54   3967 12.8
     70-74   3593 11.6
     45-49   3508 11.4
     40-44   2623  8.5
     75-79   1908  6.2
     80-84    950  3.1
       NaN    654  2.1
     35-39    122  0.4
     30-34     37  0.1



# **Step 4. Match**

In [12]:
# ── Config ────────────────────────────────────────────────────────────────────
RANDOM_SEED = 42
N_MATCH     = 8  # normals per cancer patient

In [13]:
# ── Feature tables ────────────────────────────────────────────────────
c_feat = c_combo.merge(mo_cancer_ehr[['PATIENT_STUDY_ID', 'PatientAge', 'age_bin', 'race']], on='PATIENT_STUDY_ID', how='left')
n_feat = n_combo.merge(normal_candidate_ehr[['PATIENT_STUDY_ID', 'PatientAge', 'age_bin', 'race']], on='PATIENT_STUDY_ID', how='left')

def make_stratum(df):
    return df['birads_combo'].astype(str) + '|' + df['race'].astype(str) + '|' + df['age_bin'].astype(str)

c_feat['stratum'] = make_stratum(c_feat)
n_feat['stratum'] = make_stratum(n_feat)

# ── Filter out unresolvable cancer entries before matching ────────────────────
c_feat_match = c_feat[
    c_feat['age_bin'].notna() &
    (c_feat['race'] != 'OTHER/UNKNOWN') &
    (c_feat['birads_combo'].isin(['1+1', '2+2']))
].copy()

print(f"Cancer patients used for matching : {len(c_feat_match)} / {len(c_feat)}")
print(f"Excluded (NaN age or unknown race): {len(c_feat) - len(c_feat_match)}")

Cancer patients used for matching : 240 / 253
Excluded (NaN age or unknown race): 13


In [14]:
c_feat_match['birads_combo'].unique(), c_feat_match['age_bin'].unique(), c_feat_match['race'].unique()

(array(['1+1', '2+2'], dtype=object),
 ['45-49', '40-44', '65-69', '55-59', '70-74', '60-64', '50-54', '80-84', '75-79', '35-39']
 Categories (11, object): ['30-34' < '35-39' < '40-44' < '45-49' ... '65-69' < '70-74' < '75-79' < '80-84'],
 array(['ASIAN', 'WHITE', 'BLACK'], dtype=object))

In [15]:
# ── Match ─────────────────────────────────────────────────────────────
rng = np.random.default_rng(RANDOM_SEED)
normal_pool = n_feat.copy()
normal_pool_by_stratum = {s: grp.index.tolist() for s, grp in normal_pool.groupby('stratum')}

matched_cancer_rows, matched_normal_rows, unmatched_cancer_ids = [], [], []
used_normal_indices = set()

for _, cancer_row in c_feat_match.iterrows():
    stratum   = cancer_row['stratum']
    available = [i for i in normal_pool_by_stratum.get(stratum, []) if i not in used_normal_indices]

    # Fallback: relax age bin ±5 then ±10 years if stratum is empty
    if len(available) == 0:
        birads, race, age_bin = stratum.split('|')
        age_idx = AGE_LABELS.index(age_bin) if age_bin in AGE_LABELS else -1
        for offset in [-1, 1, -2, 2]:
            adj_idx = age_idx + offset
            if 0 <= adj_idx < len(AGE_LABELS):
                adj_stratum = f'{birads}|{race}|{AGE_LABELS[adj_idx]}'
                available += [i for i in normal_pool_by_stratum.get(adj_stratum, []) if i not in used_normal_indices]

    if len(available) == 0:
        unmatched_cancer_ids.append(cancer_row['PATIENT_STUDY_ID'])
        continue

    chosen_idx = rng.choice(available, size=min(N_MATCH, len(available)), replace=False).tolist()
    used_normal_indices.update(chosen_idx)
    matched_cancer_rows.append(cancer_row)
    for idx in chosen_idx:
        row = normal_pool.loc[idx].copy()
        row['matched_cancer_id'] = cancer_row['PATIENT_STUDY_ID']
        matched_normal_rows.append(row)

matched_cancer_list  = pd.DataFrame(matched_cancer_rows).reset_index(drop=True)
matched_normal_list = pd.DataFrame(matched_normal_rows).reset_index(drop=True)

# ── Summary ───────────────────────────────────────────────────────────
print(f"Cancer matched    : {len(matched_cancer_list)} / {len(c_feat_match)}")
print(f"Normals selected  : {len(matched_normal_list)}  (ratio {len(matched_normal_list)/len(matched_cancer_list):.2f}:1)")
print(f"Unmatched IDs     : {unmatched_cancer_ids}")

Cancer matched    : 240 / 240
Normals selected  : 1920  (ratio 8.00:1)
Unmatched IDs     : []


In [16]:
matched_normal_list

,PATIENT_STUDY_ID,birads_combo,PatientAge,age_bin,race,stratum,matched_cancer_id
0,4333035247,1+1,45.0,45-49,ASIAN,1+1|ASIAN|45-49,4330421371
1,4335019106,1+1,45.0,45-49,ASIAN,1+1|ASIAN|45-49,4330421371
2,4335764139,1+1,49.0,45-49,ASIAN,1+1|ASIAN|45-49,4330421371
3,4334592605,1+1,48.0,45-49,ASIAN,1+1|ASIAN|45-49,4330421371
4,4337773749,1+1,48.0,45-49,ASIAN,1+1|ASIAN|45-49,4330421371
...,...,...,...,...,...,...,...
1915,4334447298,1+1,51.0,50-54,WHITE,1+1|WHITE|50-54,4339011646
1916,4333176835,1+1,50.0,50-54,WHITE,1+1|WHITE|50-54,4339011646
1917,4333655684,1+1,50.0,50-54,WHITE,1+1|WHITE|50-54,4339011646
1918,4333899434,1+1,51.0,50-54,WHITE,1+1|WHITE|50-54,4339011646


In [22]:
file_path = os.path.join("/Users/tracyliu/Library/CloudStorage/OneDrive-UniversityofPittsburgh/R01-MO-DBT/MO-DBT-data-curation/Data", "Control", "matched_correspondance" + ".xlsx")
matched_normal_list.to_excel(file_path, index=False)

In [17]:
matched_normal_ehr = normal_candidate_ehr[
    normal_candidate_ehr['PATIENT_STUDY_ID'].isin(matched_normal_list['PATIENT_STUDY_ID'])
].copy()

print(f"Matched normal EHR rows : {len(matched_normal_ehr)}")
print(f"Unique patients         : {matched_normal_ehr['PATIENT_STUDY_ID'].nunique()}")

Matched normal EHR rows : 1920
Unique patients         : 1920


In [18]:
matched_normal_ehr

,PATIENT_STUDY_ID,ACCESSION_NUMBER,EXAM_COMPLETED_DATE,Study,Index,DATE_TAKEN,WEIGHT,WEIGHT_UNIT,HEIGHT,HEIGHT_UNIT,BMI,AGE_MENARCHE,AGE_FIRST_LIVE_BIRTH,AGE_MENOPAUSE,AGE_HYSTERECTOMY,AGE_RIGHT_OVARY_REMOVAL,AGE_LEFT_OVARY_REMOVAL,PARITY_COUNT,PREGNANCY_COUNT,LAST_MENSTRUAL_DATE,MENSTRUAL_STATUS_CD,RISK_FACTOR_No family history of breast cancer,RISK_FACTOR_Personal breast cancer history,"RISK_FACTOR_Weak family history of breast cancer (aunt, grandmother, cousin)",RISK_FACTOR_Late child bearing (after 30),"RISK_FACTOR_Very strong family history of breast cancer (mother, sister, daughter, pre-menopause or multiple post-menopausal first degree relatives)","RISK_FACTOR_Family history of ovarian cancer in mother, sister, or daughter",RISK_FACTOR_Post-menopausal patient,RISK_FACTOR_BRCA1 gene mutation,RISK_FACTOR_Nulliparous,RISK_FACTOR_Family history of ovarian cancer in distant blood relatives,"RISK_FACTOR_Intermediate family history of breast cancer (mother, sister, daughter, post-menopause)","RISK_FACTOR_History of high risk lesion on previous biopsy, LCIS, atypical hyperplasia",RISK_FACTOR_Previous chest radiation therapy,RISK_FACTOR_Other family history of breast cancer,RISK_FACTOR_BRCA2 gene mutation,RISK_FACTOR_Family history unknown for breast cancer,RISK_FACTOR_History of ovarian cancer,RISK_FACTOR_History of endometrial cancer,RISK_FACTOR_Not available,BIRTH_DATE,GENDER_TITLE,RACE_KOREAN,RACE_WHITE,RACE_INDIAN (ASIAN),RACE_BLACK,RACE_UNREPORTED/CHOSE NOT TO DISCLOSE,RACE_OTHER ASIAN,RACE_VIETNAMESE,RACE_CHINESE,RACE_AMERICAN INDIAN/ALASKA NATIVE,RACE_FILIPINO,RACE_OTHER,RACE_JAPANESE,RACE_HAWAIIAN,RACE_OTHER PACIFIC ISLANDER,ETHNIC_NOT HISPANIC OR LATINO,"ETHNIC_ANOTHER HISPANIC, LATINO, OR SPANISH ORIGIN",ETHNIC_UNREPORTED/CHOSE NOT TO DISCLOSE,ETHNIC_HISPANIC OR LATINO,"ETHNIC_MEXICAN, MEXICAN AMERICAN, CHICANO",PatientAge,race,age_bin
7,4330039944,66820164,2021-05-21,SCREEN,INDEX-1,2021-07-30,2380.8,OZ,68.5,IN,22.30,13.0,17.0,0.0,0.0,0.0,0.0,2.0,3.0,2020-05-01,PRE,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1976-07-01,FEMALE,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,44.0,WHITE,40-44
60,4330260631,61589587,2020-01-28,SCREEN,INDEX-1,2020-01-10,2128.0,OZ,65.0,IN,22.13,11.0,49.0,50.0,0.0,0.0,0.0,1.0,1.0,NaN,POSTNAT,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1958-07-01,FEMALE,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,61.0,WHITE,60-64
74,4330307156,78758517,2018-05-30,SCREEN,INDEX-1,2021-04-18,NaN,KG,NaN,CM,55.80,15.0,0.0,49.0,0.0,0.0,0.0,0.0,NaN,NaN,POSTNAT,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1966-07-01,FEMALE,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,51.0,WHITE,50-54
141,4330351238,453366972,2021-12-15,SCREEN,INDEX-1,2018-12-12,3344.0,OZ,70.0,IN,29.99,13.0,35.0,0.0,0.0,0.0,0.0,1.0,1.0,NaN,PRE,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1969-07-01,FEMALE,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,52.0,WHITE,50-54
151,4330362606,69021057,2020-08-06,SCREEN,INDEX-1,2020-07-23,2377.6,OZ,69.5,IN,21.63,16.0,31.0,50.0,0.0,0.0,0.0,2.0,2.0,NaN,POSTNAT,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1956-07-01,FEMALE,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,64.0,WHITE,60-64
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30760,4339030964,455714740,2021-10-08,SCREEN,INDEX-1,2021-07-01,2672.0,OZ,69.0,IN,24.66,16.0,0.0,46.0,0.0,0.0,0.0,0.0,0.0,NaN,POSTNAT,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1966-07-01,FEMALE,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0

In [19]:
matched_normal = normal_candidate[
    normal_candidate['PATIENT_STUDY_ID'].isin(matched_normal_list['PATIENT_STUDY_ID'])
].copy()

print(f"Matched normal EHR rows : {len(matched_normal)}")
print(f"Unique patients         : {matched_normal['PATIENT_STUDY_ID'].nunique()}")

Matched normal EHR rows : 7601
Unique patients         : 1920


In [20]:
matched_normal

,PATIENT_STUDY_ID,ACCESSION_NUMBER,COMPOSITION_NAME,FINDING_CATEGORY,FINDING_REC,EXAM_COMPLETED_DATE,duplicate_count,EXAM_NAME,Study,Series,SIDE,Index,time_to_INDEX_days,time_to_INDEX_months
26,4330039944,66820164,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2021-05-21,2,SCREENING MAMMOGRAPHY DIGITAL BILATERAL W TOMO,SCREEN,DBT,L,INDEX-1,369.0,12.1
27,4330039944,66820164,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2021-05-21,2,SCREENING MAMMOGRAPHY DIGITAL BILATERAL W TOMO,SCREEN,DBT,R,INDEX-1,369.0,12.1
28,4330039944,452759833,Heterogeneously dense (51% - 75%),1 - Negative,N-Normal interval follow-up,2022-05-25,2,SCREENING MAMMOGRAPHY DIGITAL BILATERAL W TOMO,SCREEN,DBT,L,INDEX,NaN,NaN
29,4330039944,452759833,Heterogeneously dense (51% - 75%),1 - Negative,N-Normal interval follow-up,2022-05-25,2,SCREENING MAMMOGRAPHY DIGITAL BILATERAL W TOMO,SCREEN,DBT,R,INDEX,NaN,NaN
216,4330260631,61589587,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2020-01-28,2,SCREENING MAMMOGRAPHY DIGITAL BILATERAL W TOMO,SCREEN,DBT,L,INDEX-1,794.0,26.1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
118466,4339561039,451256756,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2022-07-09,2,SCREENING MAMMOGRAPHY DIGITAL BILATERAL W TOMO,SCREEN,DBT,R,INDEX,NaN,NaN
118531,4339605126,67949683,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2021-02-01,2,SCREENING MAMMOGRAPHY DIGITAL BILATERAL W TOMO,SCREEN,DBT,L,INDEX-1,376.0,12.4
118532,4339605126,67949683,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2021-02-01,2,SCREENING MAMMOGRAPHY DIGITAL BILATERAL W TOMO,SCREEN,DBT,R,INDEX-1,376.0,12.4
118533,4339605126,453091158,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2022-02-12,2,SCREENING MAMMOGRAPHY DIGITAL BILATERAL W TOMO,SCREEN,DBT,L,INDEX,NaN,NaN


In [21]:
file_path = os.path.join("/Users/tracyliu/Library/CloudStorage/OneDrive-UniversityofPittsburgh/R01-MO-DBT/MO-DBT-data-curation/Data", "Control", "matched_normal_ehr_@_index_1" + ".xlsx")
matched_normal_ehr.to_excel(file_path, index=False)
file_path = os.path.join("/Users/tracyliu/Library/CloudStorage/OneDrive-UniversityofPittsburgh/R01-MO-DBT/MO-DBT-data-curation/Data", "Control", "matched_normal" + ".xlsx")
matched_normal.to_excel(file_path, index=False)